Project 2 


In [3]:
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1
import part2

import nltk
import numpy as np

## Data
SST-2: sentences labelled negative (0) or positive (1). We use the train split for
cross-validation; the official test labels are hidden.

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


## Pre-trained embeddings and tokenizer

In [4]:
# 30 seconds
trained_emb_model_glovevwikigiga50d = part2.clslib.PretrainedEmbeddings(
    "glove.2024.wikigiga.50d.zip",
    50,
    skip_first_line=False,
)


In [ ]:
# 3 min run, can crash then just run with restarted kernel (keeps chrashing on my pc)
#trained_emb_model_gloveCC300d = part2.clslib.PretrainedEmbeddings(
#    "glove.42B.300d.zip",
#    300,
#    skip_first_line=False,
#)


In [5]:
# chosen model
trained_emb_model = trained_emb_model_glovevwikigiga50d
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}") 
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1287615, 50)
Size of vocabulary: 1287615
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:the
3:,
4:.
5:of
6:to
7:and
8:in
9:a


In [ ]:
train_texts = dataset['train']['sentence']

tokenizer = part2.clslib.BasicTokenizer(
    lowercase=True, 
    remove_punct=True, 
    remove_stopwords=True, 
    language='english',
    vocab=trained_emb_model.vocab
)

unk_id = tokenizer.vocab.default_index

sentences_with_unk = 0
unk_token_occurrences = 0

for text in train_texts:
    ids = tokenizer.ids(text)
    sentences_with_unk += unk_id in ids
    unk_token_occurrences += ids.count(unk_id)

print("Sentences containing at least one UNK:", sentences_with_unk)
print("Total UNK token occurrences:", unk_token_occurrences)

Sentences containing at least one UNK: 494
Total UNK token occurrences: 503


## Comparing embedding vocabulary coverage

Vocabulary coverage was compared by counting how often each embedding vocabulary failed to contain a token in the training sentences. Two counts are reported:

- **Sentences with at least one UNK:** sentences containing one or more out-of-vocabulary tokens.
- **UNK token occurrences:** total out-of-vocabulary tokens across the dataset.

| Embedding model | Sentences with at least one UNK | UNK token occurrences |
|---|---:|---:|
| GloVe 50d (Wiki + Gigaword) | 494 | 503 |
| GloVe 300d (Common Crawl) | 616 | 634 |
| fastText | 973 | 991 |
| Word2Vec | 22,584 | 31,117 |

### Observations

- GloVe 300d (Common Crawl) had the fewest UNK tokens.
- GloVe 50d (wikigiga) also had relatively few UNK tokens.
- fastText had more UNK tokens than either GloVe model.
- Word2Vec had substantially more UNK tokens than the other embeddings.

### Conclusion

Based on vocabulary coverage, both GloVe models will be used for further evaluation. The Glove embeddings are taken from [here](https://github.com/stanfordnlp/GloVe)



In [11]:
def cos_similarity(u, v):  
    """
    Return the cosine similarity (a real number) of the two words in the embedding space of our model.
    """   

    dot = np.dot(u, v)
    norm = np.linalg.norm(u) * np.linalg.norm(v)
    return dot / norm

def topk_ids(v, E, k=10, exclude_ids=()):
    v = np.asarray(v)
    E = np.asarray(E)

    v_norm = np.linalg.norm(v)
    if v_norm == 0:
        raise ValueError("The query word has a zero vector.")

    row_norms = np.linalg.norm(E, axis=1)
    valid = row_norms > 0

    scores = np.full(E.shape[0], -np.inf)
    scores[valid] = (E[valid] @ v) / (row_norms[valid] * v_norm)

    scores[list(exclude_ids)] = -np.inf

    k = min(k, np.isfinite(scores).sum())
    return np.argsort(scores)[-k:][::-1]

In [12]:
words = ['bad', 'horrible', 'neutral', 'amazing', 'terrific', 'good']
for word1 in words:
    topkidx = topk_ids(trained_emb_model.embed_word(word1), trained_emb_model.vectors)
    for idx in topkidx:
        print(f'Top 10 words for {word1}, {trained_emb_model.word(idx)}')

Top 10 words for bad, bad
Top 10 words for bad, worse
Top 10 words for bad, too
Top 10 words for bad, really
Top 10 words for bad, little
Top 10 words for bad, nothing
Top 10 words for bad, unfortunately
Top 10 words for bad, gone
Top 10 words for bad, ?
Top 10 words for bad, things
Top 10 words for horrible, horrible
Top 10 words for horrible, terrible
Top 10 words for horrible, awful
Top 10 words for horrible, dreadful
Top 10 words for horrible, horrific
Top 10 words for horrible, horrendous
Top 10 words for horrible, shocking
Top 10 words for horrible, horrifying
Top 10 words for horrible, tragic
Top 10 words for horrible, scary
Top 10 words for neutral, neutral
Top 10 words for neutral, acceptable
Top 10 words for neutral, sufficiently
Top 10 words for neutral, therefore
Top 10 words for neutral, strictly
Top 10 words for neutral, generally
Top 10 words for neutral, position
Top 10 words for neutral, determined
Top 10 words for neutral, transparent
Top 10 words for neutral, otherwi

In [15]:
u = trained_emb_model.embed_word('neutral') + trained_emb_model.embed_word('not')
topkidx = topk_ids(u, trained_emb_model.vectors)
for idx in topkidx:
    print(f'Top 10 words for u, {trained_emb_model.word(idx)}')

Top 10 words for u, not
Top 10 words for u, be
Top 10 words for u, should
Top 10 words for u, must
Top 10 words for u, if
Top 10 words for u, any
Top 10 words for u, nor
Top 10 words for u, neither
Top 10 words for u, either
Top 10 words for u, longer
